# Sorro offline council: train, check, publish

Trains Sorro's three local models on synthetic data, shrinks each to q4 ONNX for the app, scores it on held-out cases, and publishes it to Hugging Face **only if it passes its bar** and loads in transformers.js.

| Model | Base | Bar |
|---|---|---|
| router | SmolLM2-135M-Instruct | 95% correct route |
| checker | SmolLM2-360M-Instruct | catches 80% of wrong writes, under 10% false flags |
| actor | Qwen2.5-0.5B-Instruct | 85% exact tool name and valid args |

**Before you run**
1. Runtime → Change runtime type → **T4 GPU**.
2. Make a Hugging Face account and a **write** token (huggingface.co → Settings → Access Tokens). In Colab, open the key icon on the left, add a secret named `HF_TOKEN`, and allow this notebook to use it.
3. On your PC: `python training/generate.py && python training/pack.py`, which makes `training/upload.zip`.
4. Set `HF_USER` in the next cell, then Runtime → **Run all**. Upload `upload.zip` when asked.

Expect about 1 to 2 hours. The actor runs last, so a timeout loses the least.

In [ ]:
# Settings
HF_USER = ''            # your Hugging Face username, e.g. 'jhems'
PUSH = True             # False to train and score without publishing
MODELS = {
    'router':  dict(base='HuggingFaceTB/SmolLM2-135M-Instruct', epochs=3, lr=2e-4, r=16, batch=16, accum=1, max_len=512,  new_tokens=40),
    'checker': dict(base='HuggingFaceTB/SmolLM2-360M-Instruct', epochs=3, lr=2e-4, r=16, batch=8,  accum=2, max_len=768,  new_tokens=40),
    'actor':   dict(base='Qwen/Qwen2.5-0.5B-Instruct',          epochs=2, lr=1e-4, r=32, batch=2,  accum=8, max_len=2048, new_tokens=200),
}
BARS = {'router': 0.95, 'actor': 0.85, 'checker': (0.80, 0.10)}
if not HF_USER and PUSH:
    raise SystemExit('Set HF_USER first (your Hugging Face username), or set PUSH = False.')

In [ ]:
# Versions current on 2026-11-05. transformers.js is pinned to the version the Sorro app runs.
!pip install -q transformers==5.22.1 peft==0.21.2 trl==1.17.1 accelerate==1.15.0 datasets==5.1.0 optimum==2.3.0 optimum-onnx==0.1.0 onnx==1.23.2 onnxruntime==1.31.1 huggingface_hub==2.4.0
!npm install --silent --prefix /content/smoke @huggingface/transformers@3.8.1

In [ ]:
# Upload upload.zip (from python training/pack.py)
import json, os, zipfile
from google.colab import files
up = files.upload()
zipfile.ZipFile(next(iter(up)), 'r').extractall('/content/data')
D = '/content/data'
PROMPTS = json.load(open(f'{D}/prompts.json'))
# Gemini-style schemas → the JSON-schema tool list chat templates expect.
TOOLS = [{'type': 'function', 'function': t} for t in json.load(open(f'{D}/tools.json'))]
load = lambda name: [json.loads(l) for l in open(f'{D}/{name}.jsonl', encoding='utf-8')]
print({n: len(load(n)) for n in ['router.train', 'router.eval', 'actor.train', 'actor.eval', 'checker.train', 'checker.eval']})

In [ ]:
# Train: LoRA on the assistant turns only (prompt/completion split), then merge into the base in fp32.
import torch, gc
from datasets import Dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

def split_row(r, tok, tools):
    # Rendered to text here: a Dataset of nested dicts would merge every tool's argument keys and fill the gaps with null.
    k = next(i for i, m in enumerate(r['messages']) if m['role'] == 'assistant')
    prompt = tok.apply_chat_template(r['messages'][:k], tools=tools, add_generation_prompt=True, tokenize=False)
    full = tok.apply_chat_template(r['messages'], tools=tools, tokenize=False)
    assert full.startswith(prompt), 'chat template: the prompt is not a prefix of the full conversation'
    return {'prompt': prompt, 'completion': full[len(prompt):]}

def train(name):
    c = MODELS[name]
    tok = AutoTokenizer.from_pretrained(c['base'])
    tools = TOOLS if name == 'actor' else None
    rows = [split_row(r, tok, tools) for r in load(f'{name}.train')]
    assert 'null' not in rows[0]['completion'], rows[0]['completion'][:400]
    print(name, 'first example:', (rows[0]['prompt'] + rows[0]['completion'])[-600:])
    # fp32 weights; fp16=True below still trains in mixed precision without the fp16 overflow risk.
    model = AutoModelForCausalLM.from_pretrained(c['base'], dtype=torch.float32, device_map='cuda')
    cfg = SFTConfig(output_dir=f'/content/out/{name}/lora', num_train_epochs=c['epochs'], learning_rate=c['lr'],
                    per_device_train_batch_size=c['batch'], gradient_accumulation_steps=c['accum'], max_length=c['max_len'],
                    fp16=True, logging_steps=25, save_strategy='no', report_to='none', completion_only_loss=True)
    lora = LoraConfig(r=c['r'], lora_alpha=2 * c['r'], lora_dropout=0.05, target_modules='all-linear', task_type='CAUSAL_LM')
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=Dataset.from_list(rows), processing_class=tok, peft_config=lora)
    trainer.train()
    trainer.model.save_pretrained(f'/content/out/{name}/lora')  # the PeftModel, so this is the adapter
    del model, trainer; gc.collect(); torch.cuda.empty_cache()
    base = AutoModelForCausalLM.from_pretrained(c['base'], dtype=torch.float32)
    merged = PeftModel.from_pretrained(base, f'/content/out/{name}/lora').merge_and_unload()
    merged.save_pretrained(f'/content/out/{name}/merged')
    tok.save_pretrained(f'/content/out/{name}/merged')
    del base, merged; gc.collect()

In [ ]:
# Export to ONNX and quantise to q4, in the layout transformers.js loads with dtype 'q4': <name>/onnx/model_q4.onnx
import shutil, subprocess, onnx
from onnxruntime.quantization.matmul_nbits_quantizer import MatMulNBitsQuantizer

def export(name):
    src, out = f'/content/out/{name}/merged', f'/content/out/{name}/release'
    raw = f'/content/out/{name}/onnx_fp32'
    subprocess.run(['optimum-cli', 'export', 'onnx', '--model', src, '--task', 'text-generation-with-past', raw], check=True)
    q = MatMulNBitsQuantizer(onnx.load(f'{raw}/model.onnx'), block_size=32, is_symmetric=True)
    q.process()
    os.makedirs(f'{out}/onnx', exist_ok=True)
    q.model.save_model_to_file(f'{out}/onnx/model_q4.onnx')  # every q4 model here is far under the 2 GB single-file limit
    for f in os.listdir(raw):  # tokenizer, config, generation_config, chat template
        if not f.endswith(('.onnx', '.onnx_data', '.onnx.data')):
            shutil.copy(f'{raw}/{f}', out)
    # transformers.js 3.8 reads the chat template from tokenizer_config.json, not chat_template.jinja.
    tc = json.load(open(f'{out}/tokenizer_config.json'))
    if not tc.get('chat_template') and os.path.exists(f'{out}/chat_template.jinja'):
        tc['chat_template'] = open(f'{out}/chat_template.jinja').read()
        json.dump(tc, open(f'{out}/tokenizer_config.json', 'w'), indent=2)
    # A flat copy for optimum's ORT loader, which expects model.onnx beside config.json.
    ort = f'/content/out/{name}/ort'
    shutil.copytree(out, ort, dirs_exist_ok=True, ignore=shutil.ignore_patterns('onnx'))
    shutil.copy(f'{out}/onnx/model_q4.onnx', f'{ort}/model.onnx')
    print(name, 'q4 MB:', round(sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(f'{out}/onnx') for f in fs) / 1e6))

In [ ]:
# Score the q4 model on the held-out eval, greedy. This is the model the app ships, not the fp32 one.
import re
from optimum.onnxruntime import ORTModelForCausalLM

def clean(d):
    return {k: v for k, v in d.items() if v not in (None, '', [], {})}

def first_call(text):
    m = re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', text, re.S)
    try:
        return json.loads(m.group(1)) if m else None
    except json.JSONDecodeError:
        return None

def evaluate(name):
    c, path = MODELS[name], f'/content/out/{name}/release'
    tok = AutoTokenizer.from_pretrained(path)
    model = ORTModelForCausalLM.from_pretrained(f'/content/out/{name}/ort')  # the same q4 file, flat layout
    hits, wrong_caught, wrong_n, ok_flagged, ok_n, misses = 0, 0, 0, 0, 0, []
    rows = load(f'{name}.eval')
    for r in rows:
        if name == 'actor':
            prompt = tok.apply_chat_template(r['messages'][:r['expect']['at']], tools=TOOLS, add_generation_prompt=True, tokenize=False)
        else:
            prompt = tok.apply_chat_template(r['messages'][:-1], add_generation_prompt=True, tokenize=False)
        ids = tok(prompt, return_tensors='pt')
        out = tok.decode(model.generate(**ids, max_new_tokens=c['new_tokens'], do_sample=False)[0][ids['input_ids'].shape[1]:], skip_special_tokens=False)
        out = out.split(tok.eos_token)[0].strip() if tok.eos_token else out.strip()
        if name == 'router':
            got = 'chat' if out.startswith('chat:') else out.split()[0] if out else ''
            ok = got == r['expect']['route']
        elif name == 'actor':
            call = first_call(out)
            ok = bool(call) and call.get('name') == r['expect']['name'] and clean(call.get('arguments') or {}) == clean(r['expect']['arguments'])
        else:
            flagged = out.lower().startswith('wrong')
            if r['expect']['verdict'] == 'wrong':
                wrong_n += 1; wrong_caught += flagged
            else:
                ok_n += 1; ok_flagged += flagged
            ok = flagged == (r['expect']['verdict'] == 'wrong')
        hits += ok
        if not ok and len(misses) < 8:
            misses.append({'want': r['expect'], 'got': out[:200]})
    if name == 'checker':
        score = {'catch_rate': wrong_caught / wrong_n, 'false_flag_rate': ok_flagged / ok_n}
        passed = score['catch_rate'] >= BARS['checker'][0] and score['false_flag_rate'] < BARS['checker'][1]
    else:
        score = {'accuracy': hits / len(rows)}
        passed = score['accuracy'] >= BARS[name]
    report = {'model': name, 'base': c['base'], 'eval_cases': len(rows), **score, 'bar': BARS[name], 'passed': passed}
    json.dump(report, open(f'{path}/sorro_eval.json', 'w'), indent=2)
    print(json.dumps(report, indent=2)); print('sample misses:', json.dumps(misses, indent=1)[:2000])
    return passed

In [ ]:
# Smoke test: load the release folder with the same transformers.js the app uses, and generate once.
SMOKE = r"""
import { env, pipeline } from '@huggingface/transformers';
env.allowLocalModels = true; env.allowRemoteModels = false; env.localModelPath = '/content/out/';
const [name, msgs] = [process.argv[2], JSON.parse(process.argv[3])];
const gen = await pipeline('text-generation', `${name}/release`, { dtype: 'q4', device: 'cpu' });
const out = await gen(msgs, { max_new_tokens: 40, do_sample: false });
console.log('smoke ok:', JSON.stringify(out[0].generated_text.at(-1).content).slice(0, 200));
"""
open('/content/smoke/smoke.mjs', 'w').write(SMOKE)

def smoke(name):
    r = load(f'{name}.eval')[0]
    msgs = r['messages'][:r['expect']['at']] if name == 'actor' else r['messages'][:-1]
    res = subprocess.run(['node', '/content/smoke/smoke.mjs', name, json.dumps(msgs)], cwd='/content/smoke', capture_output=True, text=True)
    print(res.stdout[-400:], res.stderr[-800:])
    return res.returncode == 0 and 'smoke ok' in res.stdout

In [ ]:
# Publish only what passed both gates.
from huggingface_hub import HfApi
from google.colab import userdata

def publish(name, passed, smoked):
    if not (PUSH and passed and smoked):
        print(f'{name}: NOT published (bar passed={passed}, smoke passed={smoked}, PUSH={PUSH})'); return None
    api, repo = HfApi(token=userdata.get('HF_TOKEN')), f'{HF_USER}/sorro-{name}'
    rep = json.load(open(f'/content/out/{name}/release/sorro_eval.json'))
    open(f'/content/out/{name}/release/README.md', 'w').write(
        f"---\nlibrary_name: transformers.js\nbase_model: {MODELS[name]['base']}\n---\n# sorro-{name}\n\n"
        f"Sorro's offline {name}, fine-tuned from {MODELS[name]['base']} with LoRA and quantised to q4 ONNX.\n\n"
        f"Trained on synthetic data only, generated from templates. No personal data.\n\nEval: `{json.dumps({k: v for k, v in rep.items() if k not in ('model', 'base')})}`\n")
    api.create_repo(repo, exist_ok=True)
    info = api.upload_folder(repo_id=repo, folder_path=f'/content/out/{name}/release', commit_message='Sorro offline model')
    print(f'{name}: published https://huggingface.co/{repo} at {info.oid}')
    return repo, info.oid

In [ ]:
# Run: router, checker, then actor (the largest last).
results = {}
for name in ['router', 'checker', 'actor']:
    train(name); export(name)
    passed = evaluate(name); smoked = smoke(name)
    results[name] = {'passed': passed, 'smoke': smoked, 'published': publish(name, passed, smoked)}
print(json.dumps(results, indent=2))

## Send back to Claude
The `results` printed above: each model's repo name and commit (`published`), and its `sorro_eval.json` scores. B3b pins those exact commits in the app.

A model that missed its bar is not published. Its sample misses show what to fix in `training/generate.py`. Adjust, regenerate, re-pack, and rerun only that model.